# XAI aplicado al riesgo de morosidad
## Modelo supervisado y Multi-Armed Bandit Contextual

**Escenario:** sistema explicable de alerta temprana para identificar clientes con riesgo de sufrir una morosidad de 90 días o más durante los próximos dos años.

El objetivo operativo no es rechazar automáticamente una operación, sino decidir entre dos acciones:

- **Acción 0 — seguimiento habitual.**
- **Acción 1 — revisión o intervención preventiva.**

No detectar una futura morosidad (falso negativo) se considera diez veces más costoso que revisar innecesariamente a un cliente (falso positivo).

### Objetivos del notebook

1. Analizar y preparar los datos de riesgo crediticio.
2. Entrenar un modelo supervisado clásico como *baseline*.
3. Seleccionar el umbral de decisión minimizando un coste de negocio asimétrico.
4. Explicar el modelo mediante importancia por permutación, ejemplos locales y un árbol subrogado.
5. Formular el mismo problema como un Multi-Armed Bandit Contextual e implementar LinUCB.
6. Explicar la política aprendida por el bandit mediante un segundo árbol subrogado.
7. Generar decisiones para el conjunto de producción.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def localizar_archivo(nombre):
    candidatos = [Path(nombre), Path("upload") / nombre, Path("/content") / nombre]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No se encuentra {nombre}. Colócalo junto al notebook o en la carpeta upload/."
    )

ruta_construccion = localizar_archivo("cs_construccion.csv")
ruta_produccion = localizar_archivo("cs_produccion.csv")
ruta_diccionario = localizar_archivo("DataDictionary.csv")

df_raw = pd.read_csv(ruta_construccion)
df_produccion_raw = pd.read_csv(ruta_produccion)
diccionario = pd.read_csv(ruta_diccionario, sep=";").drop(
    columns=["Unnamed: 0"], errors="ignore"
)

print("Construcción:", df_raw.shape)
print("Producción:", df_produccion_raw.shape)
display(diccionario)

## 1. Comprensión y calidad de los datos

La variable `SeriousDlqin2yrs` vale 1 cuando la persona experimenta una morosidad de 90 días o más en los dos años siguientes. El conjunto de construcción contiene las etiquetas; en producción la columna está vacía y solo se utilizará para generar decisiones.

In [ ]:
display(df_raw.head())
display(df_raw.describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

distribucion = df_raw["SeriousDlqin2yrs"].value_counts(normalize=True).sort_index()
print("Distribución de la variable objetivo:")
display((100 * distribucion).rename("porcentaje").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_raw["SeriousDlqin2yrs"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"]
)
axes[0].set(title="Distribución de clases", xlabel="Morosidad grave", ylabel="Clientes")
(100 * df_raw.isna().mean()).sort_values(ascending=False).plot(kind="bar", ax=axes[1])
axes[1].set(title="Valores ausentes", ylabel="Porcentaje")
plt.tight_layout()
plt.show()

In [ ]:
columnas_retrasos = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

anomalias_retrasos = (df_raw[columnas_retrasos] >= 96).all(axis=1)
print("Filas con los códigos anómalos 96/98 en los tres retrasos:", anomalias_retrasos.sum())
print("Edades no válidas (<= 0):", (df_raw["age"] <= 0).sum())

def limpiar_variables(df):
    # Conserva las filas y convierte códigos anómalos en NaN. El imputador
    # añadirá indicadores de ausencia para no perder esa información.
    limpio = df.copy()
    limpio.loc[limpio["age"] <= 0, "age"] = np.nan
    for columna in columnas_retrasos:
        limpio.loc[limpio[columna] >= 96, columna] = np.nan
    return limpio

df = limpiar_variables(df_raw)
df_produccion = limpiar_variables(df_produccion_raw)

In [ ]:
columnas_numericas = [c for c in df.columns if c != "SeriousDlqin2yrs"]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for columna, ax in zip(columnas_numericas, axes.ravel()):
    datos_plot = df[columna].dropna()
    limites = datos_plot.quantile([0.01, 0.99]).values
    datos_plot.clip(*limites).hist(bins=30, ax=ax, color="#4C78A8")
    ax.set_title(columna, fontsize=9)
plt.suptitle("Distribuciones recortadas a los percentiles 1–99 para facilitar su lectura")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="vlag", center=0)
plt.title("Correlaciones lineales")
plt.tight_layout()
plt.show()

## 2. Separación y preprocesamiento

Se crean tres conjuntos estratificados:

- **Entrenamiento (60 %):** ajuste del preprocesamiento y de los modelos.
- **Validación (20 %):** selección del umbral y decisiones de diseño.
- **Prueba (20 %):** evaluación final, utilizada una sola vez.

Las medianas y la escala se aprenden exclusivamente con entrenamiento para evitar fuga de información. Se crean indicadores adicionales para las variables que contienen valores ausentes.

In [ ]:
X = df.drop(columns="SeriousDlqin2yrs")
y = df["SeriousDlqin2yrs"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

preprocesador = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), columnas_numericas)
])

X_train_s = preprocesador.fit_transform(X_train)
X_val_s = preprocesador.transform(X_val)
X_test_s = preprocesador.transform(X_test)

nombres_transformados = [
    nombre.replace("num__", "").replace("missingindicator_", "Falta_")
    for nombre in preprocesador.get_feature_names_out()
]

# Vista en unidades originales para los árboles explicativos.
columnas_con_ausencias_train = [c for c in columnas_numericas if X_train[c].isna().any()]
medianas_train = X_train[columnas_numericas].median()

def crear_vista_explicable(X_original):
    vista = X_original[columnas_numericas].copy()
    for columna in columnas_con_ausencias_train:
        vista[f"Falta_{columna}"] = vista[columna].isna().astype(int)
    vista[columnas_numericas] = vista[columnas_numericas].fillna(medianas_train)
    return vista

X_train_exp = crear_vista_explicable(X_train)
X_val_exp = crear_vista_explicable(X_val)
X_test_exp = crear_vista_explicable(X_test)
nombres_explicables = X_train_exp.columns.tolist()

print("Train / validación / test:", X_train.shape, X_val.shape, X_test.shape)
print("Variables tras añadir indicadores de ausencia:", len(nombres_transformados))

## 3. Modelo supervisado sensible al coste

Se utiliza un Random Forest con ponderación de clases. El modelo genera probabilidades, pero la decisión final depende de un umbral. Ese umbral se seleccionará en validación minimizando:

\[
\text{coste medio}=\frac{10\,FN + 1\,FP}{N}
\]

Así se refleja que omitir un cliente que posteriormente cae en morosidad es más grave que realizar una revisión innecesaria.

In [ ]:
COSTE_FP = 1.0
COSTE_FN = 10.0

modelo_supervisado = RandomForestClassifier(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=10,
    class_weight={0: 1, 1: 10},
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
modelo_supervisado.fit(X_train_s, y_train)

prob_val = modelo_supervisado.predict_proba(X_val_s)[:, 1]

def coste_clasificacion(y_real, pred):
    y_real = np.asarray(y_real)
    pred = np.asarray(pred)
    fp = np.sum((pred == 1) & (y_real == 0))
    fn = np.sum((pred == 0) & (y_real == 1))
    return (COSTE_FP * fp + COSTE_FN * fn) / len(y_real)

umbrales = np.linspace(0.01, 0.99, 199)
costes_val = [coste_clasificacion(y_val, prob_val >= u) for u in umbrales]
umbral_optimo = float(umbrales[np.argmin(costes_val)])

plt.figure(figsize=(8, 4))
plt.plot(umbrales, costes_val)
plt.axvline(umbral_optimo, color="red", linestyle="--", label=f"Óptimo = {umbral_optimo:.3f}")
plt.xlabel("Umbral de probabilidad")
plt.ylabel("Coste medio en validación")
plt.title("Selección del umbral según el coste de negocio")
plt.legend()
plt.show()

print(f"Umbral seleccionado exclusivamente en validación: {umbral_optimo:.3f}")

In [ ]:
prob_test = modelo_supervisado.predict_proba(X_test_s)[:, 1]
pred_test_05 = (prob_test >= 0.50).astype(int)
pred_test = (prob_test >= umbral_optimo).astype(int)

def resumen_metricas(nombre, y_real, prob, pred):
    return pd.Series({
        "modelo": nombre,
        "accuracy": accuracy_score(y_real, pred),
        "balanced_accuracy": balanced_accuracy_score(y_real, pred),
        "precision": precision_score(y_real, pred, zero_division=0),
        "recall": recall_score(y_real, pred, zero_division=0),
        "f1": f1_score(y_real, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_real, prob),
        "pr_auc": average_precision_score(y_real, prob),
        "coste_medio": coste_clasificacion(y_real, pred),
    })

resultados_supervisado = pd.DataFrame([
    resumen_metricas("Random Forest (umbral 0.50)", y_test, prob_test, pred_test_05),
    resumen_metricas(f"Random Forest (umbral {umbral_optimo:.3f})", y_test, prob_test, pred_test),
])
display(resultados_supervisado.set_index("modelo").round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(confusion_matrix(y_test, pred_test), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set(title="Matriz de confusión", xlabel="Predicción", ylabel="Real")
RocCurveDisplay.from_predictions(y_test, prob_test, ax=axes[1])
axes[1].set_title("Curva ROC")
PrecisionRecallDisplay.from_predictions(y_test, prob_test, ax=axes[2])
axes[2].set_title("Curva precisión-recall")
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_test, digits=3))

### Lectura de las métricas

Debido al desbalance, el *accuracy* puede parecer alto aunque apenas se detecten clientes morosos. Por eso las medidas principales son:

- **Recall de la clase 1:** proporción de futuras morosidades detectadas.
- **PR-AUC:** calidad de la ordenación cuando la clase positiva es poco frecuente.
- **Coste medio:** impacto de los falsos negativos y positivos según el escenario.

## 4. XAI del modelo supervisado

Se combinan tres niveles de explicación:

1. **Importancia global por permutación:** cuánto empeora la PR-AUC al desordenar una variable.
2. **Árbol subrogado:** reglas sencillas que imitan las decisiones de la caja negra.
3. **Explicación local:** camino seguido por el árbol para un cliente de alto riesgo.

La fidelidad del árbol indica cuánto se parece al Random Forest; no mide su exactitud respecto a la realidad.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx_xai = rng.choice(len(X_test_s), size=min(4000, len(X_test_s)), replace=False)
perm = permutation_importance(
    modelo_supervisado,
    X_test_s[idx_xai],
    y_test.iloc[idx_xai],
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importancia = pd.DataFrame({
    "variable": nombres_transformados,
    "importancia_media": perm.importances_mean,
    "desviacion": perm.importances_std,
}).sort_values("importancia_media", ascending=False)

display(importancia.head(12))
top = importancia.head(12).sort_values("importancia_media")
plt.figure(figsize=(9, 6))
plt.barh(top["variable"], top["importancia_media"], xerr=top["desviacion"])
plt.xlabel("Descenso de PR-AUC al permutar")
plt.title("Importancia global por permutación")
plt.tight_layout()
plt.show()

In [ ]:
pred_caja_negra_val = (prob_val >= umbral_optimo).astype(int)
pred_caja_negra_test = pred_test

subrogado_supervisado = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=150, random_state=RANDOM_STATE
)
subrogado_supervisado.fit(X_val_exp, pred_caja_negra_val)
fidelidad_supervisado = subrogado_supervisado.score(X_test_exp, pred_caja_negra_test)

print(f"Fidelidad del árbol respecto al Random Forest en test: {fidelidad_supervisado:.3f}")
print("\nReglas del árbol subrogado:\n")
print(export_text(subrogado_supervisado, feature_names=nombres_explicables, decimals=2))

plt.figure(figsize=(18, 8))
plot_tree(
    subrogado_supervisado,
    feature_names=nombres_explicables,
    class_names=["Habitual", "Preventiva"],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Árbol subrogado del modelo supervisado")
plt.show()

In [ ]:
# Explicación local de un cliente con probabilidad alta.
i_local = int(np.argmax(prob_test))
x_local = X_test_exp.iloc[i_local].to_numpy()
fila_original = X_test.iloc[i_local]

print(f"Cliente seleccionado: posición {i_local} del conjunto de prueba")
print(f"Probabilidad de morosidad estimada: {prob_test[i_local]:.3f}")
print(f"Decisión de la caja negra: {'intervención preventiva' if pred_test[i_local] else 'seguimiento habitual'}")
print(f"Resultado real: {int(y_test.iloc[i_local])}")
display(fila_original.to_frame("valor"))

node_indicator = subrogado_supervisado.decision_path(x_local.reshape(1, -1))
leaf_id = subrogado_supervisado.apply(x_local.reshape(1, -1))[0]
tree_ = subrogado_supervisado.tree_

print("Camino explicativo aproximado del árbol subrogado:")
for node_id in node_indicator.indices:
    if node_id == leaf_id:
        clase = subrogado_supervisado.classes_[np.argmax(tree_.value[node_id][0])]
        print(f"  -> Hoja {node_id}: acción aproximada = {clase}")
        continue
    feature = tree_.feature[node_id]
    threshold = tree_.threshold[node_id]
    signo = "<=" if x_local[feature] <= threshold else ">"
    print(f"  {nombres_explicables[feature]} = {x_local[feature]:.2f} {signo} {threshold:.2f}")

## 5. Enfoque con Multi-Armed Bandit Contextual

Un bandit contextual observa los datos del cliente, elige una acción y recibe únicamente el refuerzo de la acción elegida.

| Situación | Acción | Refuerzo |
|---|---|---:|
| Cliente sin futura morosidad | Seguimiento habitual | 0 |
| Cliente con futura morosidad | Intervención preventiva | 0 |
| Cliente sin futura morosidad | Intervención preventiva | -1 |
| Cliente con futura morosidad | Seguimiento habitual | -10 |

Se utiliza **LinUCB**, que aprende una recompensa lineal independiente para cada acción y añade un término de incertidumbre para explorar alternativas.

> Limitación conceptual: este dataset registra morosidad, pero no el efecto causal de intervenciones reales. Por ello, el bandit reproduce aquí una decisión sensible al coste; no demuestra que una intervención evite la morosidad.

In [ ]:
ACCION_HABITUAL = 0
ACCION_PREVENTIVA = 1

def recompensa(accion, resultado_real):
    if accion == ACCION_HABITUAL and resultado_real == 1:
        return -COSTE_FN
    if accion == ACCION_PREVENTIVA and resultado_real == 0:
        return -COSTE_FP
    return 0.0

class LinUCB:
    # Bandit contextual lineal con actualización Sherman-Morrison.
    def __init__(self, n_acciones, n_features, alpha=0.7):
        self.n_acciones = n_acciones
        self.n_features = n_features
        self.alpha = alpha
        self.A_inv = np.stack([np.eye(n_features) for _ in range(n_acciones)])
        self.b = np.zeros((n_acciones, n_features))

    def scores(self, x):
        puntuaciones = np.empty(self.n_acciones)
        for a in range(self.n_acciones):
            theta = self.A_inv[a] @ self.b[a]
            incertidumbre = np.sqrt(max(x @ self.A_inv[a] @ x, 0.0))
            puntuaciones[a] = x @ theta + self.alpha * incertidumbre
        return puntuaciones

    def predict_one(self, x):
        return int(np.argmax(self.scores(x)))

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X], dtype=int)

    def update(self, x, accion, reward):
        # (A + xx')^-1 mediante Sherman-Morrison.
        Ainv_x = self.A_inv[accion] @ x
        denominador = 1.0 + x @ Ainv_x
        self.A_inv[accion] -= np.outer(Ainv_x, Ainv_x) / denominador
        self.b[accion] += reward * x

In [ ]:
# Añadimos intercepto. Se recorre entrenamiento en orden aleatorio simulando llegadas.
X_train_bandit = np.column_stack([np.ones(len(X_train_s)), X_train_s])
X_val_bandit = np.column_stack([np.ones(len(X_val_s)), X_val_s])
X_test_bandit = np.column_stack([np.ones(len(X_test_s)), X_test_s])

bandit = LinUCB(n_acciones=2, n_features=X_train_bandit.shape[1], alpha=0.7)
rng = np.random.default_rng(RANDOM_STATE)
orden = rng.permutation(len(X_train_bandit))

recompensas_online = []
acciones_online = []
for i in orden:
    contexto = X_train_bandit[i]
    accion = bandit.predict_one(contexto)
    reward = recompensa(accion, int(y_train.iloc[i]))
    bandit.update(contexto, accion, reward)
    recompensas_online.append(reward)
    acciones_online.append(accion)

coste_acumulado = -np.cumsum(recompensas_online) / np.arange(1, len(recompensas_online) + 1)
ventana = 1000
coste_movil = pd.Series(-np.array(recompensas_online)).rolling(ventana).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(coste_acumulado)
axes[0].set(title="Coste medio acumulado durante el aprendizaje", xlabel="Interacciones", ylabel="Coste")
axes[1].plot(coste_movil)
axes[1].set(title=f"Coste medio móvil ({ventana} interacciones)", xlabel="Interacciones", ylabel="Coste")
plt.tight_layout()
plt.show()

print(f"Refuerzo medio online: {np.mean(recompensas_online):.4f}")
print(f"Acciones preventivas durante entrenamiento: {100*np.mean(acciones_online):.2f}%")

In [ ]:
acciones_bandit_test = bandit.predict(X_test_bandit)

def coste_acciones(y_real, acciones):
    return -np.mean([recompensa(a, int(y)) for a, y in zip(acciones, y_real)])

rng_eval = np.random.default_rng(RANDOM_STATE)
comparacion_politicas = pd.DataFrame([
    {"política": "Siempre seguimiento habitual", "coste_medio": coste_acciones(y_test, np.zeros(len(y_test), dtype=int)), "porcentaje_preventiva": 0.0},
    {"política": "Siempre intervención preventiva", "coste_medio": coste_acciones(y_test, np.ones(len(y_test), dtype=int)), "porcentaje_preventiva": 100.0},
    {"política": "Aleatoria 50/50", "coste_medio": coste_acciones(y_test, rng_eval.integers(0, 2, len(y_test))), "porcentaje_preventiva": 50.0},
    {"política": "Supervisado + umbral de coste", "coste_medio": coste_acciones(y_test, pred_test), "porcentaje_preventiva": 100 * pred_test.mean()},
    {"política": "LinUCB contextual", "coste_medio": coste_acciones(y_test, acciones_bandit_test), "porcentaje_preventiva": 100 * acciones_bandit_test.mean()},
]).sort_values("coste_medio")

display(comparacion_politicas.round(4))

plt.figure(figsize=(9, 4))
sns.barplot(data=comparacion_politicas, x="coste_medio", y="política", palette="Blues_r")
plt.title("Comparación de políticas en el conjunto de prueba")
plt.xlabel("Coste medio (menor es mejor)")
plt.tight_layout()
plt.show()

print("Matriz: decisión LinUCB frente al resultado real")
display(pd.crosstab(pd.Series(y_test.to_numpy(), name="Morosidad real"),
                    pd.Series(acciones_bandit_test, name="Acción LinUCB"), normalize="index").round(3))

## 6. XAI de la política LinUCB

El bandit es la caja negra que se desea interpretar. Se generan sus decisiones sobre validación y se entrena un árbol pequeño para imitarlas. La fidelidad se calcula después sobre prueba, que el árbol no ha visto.

In [ ]:
acciones_bandit_val = bandit.predict(X_val_bandit)
subrogado_bandit = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=150, random_state=RANDOM_STATE
)
subrogado_bandit.fit(X_val_exp, acciones_bandit_val)
acciones_subrogado_test = subrogado_bandit.predict(X_test_exp)
fidelidad_bandit = accuracy_score(acciones_bandit_test, acciones_subrogado_test)

print(f"Fidelidad del árbol respecto a LinUCB en test: {fidelidad_bandit:.3f}")
print("\nReglas aproximadas de la política LinUCB:\n")
print(export_text(subrogado_bandit, feature_names=nombres_explicables, decimals=2))

plt.figure(figsize=(18, 8))
plot_tree(
    subrogado_bandit,
    feature_names=nombres_explicables,
    class_names=["Habitual", "Preventiva"],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Árbol subrogado de la política LinUCB")
plt.show()

In [ ]:
# Coeficientes lineales aprendidos por acción (lectura complementaria, no causal).
coeficientes = []
for accion, nombre_accion in [(0, "Habitual"), (1, "Preventiva")]:
    theta = bandit.A_inv[accion] @ bandit.b[accion]
    for variable, coef in zip(["Intercepto"] + nombres_transformados, theta):
        coeficientes.append({"acción": nombre_accion, "variable": variable, "coeficiente_reward": coef})

coef_df = pd.DataFrame(coeficientes)
for accion in ["Habitual", "Preventiva"]:
    print(f"Variables con coeficientes de mayor magnitud — acción {accion}")
    display(coef_df[coef_df["acción"] == accion]
            .assign(magnitud=lambda d: d["coeficiente_reward"].abs())
            .sort_values("magnitud", ascending=False)
            .head(8).drop(columns="magnitud"))

## 7. Aplicación al conjunto de producción

Se conservan el índice original y las variables de entrada, y se añaden:

- Probabilidad estimada por el modelo supervisado.
- Decisión supervisada con el umbral seleccionado.
- Acción propuesta por LinUCB.
- Segmento descriptivo de riesgo.

Estas decisiones requieren supervisión humana y no deberían utilizarse como único criterio para denegar financiación.

In [ ]:
X_produccion = df_produccion.drop(columns="SeriousDlqin2yrs", errors="ignore")
X_produccion_s = preprocesador.transform(X_produccion)
prob_produccion = modelo_supervisado.predict_proba(X_produccion_s)[:, 1]
decision_produccion = (prob_produccion >= umbral_optimo).astype(int)
X_produccion_bandit = np.column_stack([np.ones(len(X_produccion_s)), X_produccion_s])
accion_bandit_produccion = bandit.predict(X_produccion_bandit)

salida_produccion = df_produccion_raw.drop(columns="SeriousDlqin2yrs", errors="ignore").copy()
salida_produccion["probabilidad_morosidad_2_anios"] = prob_produccion
salida_produccion["decision_supervisada"] = decision_produccion
salida_produccion["accion_linucb"] = accion_bandit_produccion
salida_produccion["segmento_riesgo"] = pd.cut(
    prob_produccion,
    bins=[-np.inf, 0.10, 0.30, np.inf],
    labels=["bajo", "medio", "alto"],
)

salida_produccion.to_csv("cs_produccion_predicciones.csv", index=False)
display(salida_produccion.head())
print("Archivo generado: cs_produccion_predicciones.csv")
print("Distribución de decisiones supervisadas:")
display(salida_produccion["decision_supervisada"].value_counts(normalize=True).rename("proporción"))

## 8. Conclusiones y limitaciones

- El desbalance obliga a complementar el *accuracy* con recall, PR-AUC y coste medio.
- El umbral óptimo depende del coste atribuido a cada error; no es una propiedad universal del modelo.
- El Random Forest ofrece un *baseline* supervisado fuerte y el árbol subrogado lo resume mediante reglas aproximadas.
- LinUCB aprende de refuerzo parcial y permite representar exploración frente a explotación.
- La fidelidad de un subrogado mide la imitación de la caja negra, no la corrección real ni la causalidad.
- Las variables sensibles, especialmente la edad, deben someterse a revisión de sesgo, normativa y gobernanza.
- Los datos no contienen tratamientos ni resultados contrafactuales. Por tanto, no se puede afirmar que la intervención propuesta evite la morosidad.
- En un despliegue real habría que calibrar probabilidades, monitorizar deriva, revisar la estabilidad del umbral y mantener una decisión humana responsable.